# Generate Insert Annotations

RBakker 13 Feb 2025

- This notebook generates the insert annotations file for processing the linkage library

In [8]:
suppressPackageStartupMessages(library(Biostrings))
suppressPackageStartupMessages(library(plyranges))
suppressPackageStartupMessages(library(tidyverse))

wt_seq<-"ATGGACGCTCAGACTCGCCGCCGCGAGCGCCGTGCAGAAAAACAAGCCCAGTGGAAAGCCGCCAAC"
codons<-names(GENETIC_CODE)

### Generate Table of All Inserts

In [9]:
replace_codon <- function(seq, wt_codon, mut_codon, codon_position) {
  start_pos <- (codon_position - 1) * 3 + 1
  end_pos <- start_pos + 2
  str_sub(seq, start_pos, end_pos) <- mut_codon
  return(seq)
}

insert_list<- as_tibble(data.frame(seq=wt_seq))%>%
    mutate(wt_codon = map(seq, ~ str_extract_all(.x, ".{3}")[[1]])) %>%  # Split every 3 characters
    unnest(wt_codon) %>%
    mutate(codon_position = row_number())%>%
    expand_grid(mut_codon = codons) %>%
    mutate(insert_seq = replace_codon(seq, wt_codon, mut_codon, codon_position),
        wt_aa=as.character(translate(DNAStringSet(wt_codon))),
        mut_aa=as.character(translate(DNAStringSet(mut_codon))),
        insert_seq_rc=as.character(reverseComplement(DNAStringSet(insert_seq))),
    )%>%
    select(-seq)%>%
    distinct(insert_seq,.keep_all = TRUE)%>%
    mutate(row_num=row_number())%>%
    print()


# A tibble: 1,387 × 8
   wt_codon codon_position mut_codon insert_seq     wt_aa mut_aa inser…¹ row_num
   <chr>             <int> <chr>     <chr>          <chr> <chr>  <chr>     <int>
 1 ATG                   1 TTT       TTTGACGCTCAGA… M     F      GTTGGC…       1
 2 ATG                   1 TTC       TTCGACGCTCAGA… M     F      GTTGGC…       2
 3 ATG                   1 TTA       TTAGACGCTCAGA… M     L      GTTGGC…       3
 4 ATG                   1 TTG       TTGGACGCTCAGA… M     M      GTTGGC…       4
 5 ATG                   1 TCT       TCTGACGCTCAGA… M     S      GTTGGC…       5
 6 ATG                   1 TCC       TCCGACGCTCAGA… M     S      GTTGGC…       6
 7 ATG                   1 TCA       TCAGACGCTCAGA… M     S      GTTGGC…       7
 8 ATG                   1 TCG       TCGGACGCTCAGA… M     S      GTTGGC…       8
 9 ATG                   1 TAT       TATGACGCTCAGA… M     Y      GTTGGC…       9
10 ATG                   1 TAC       TACGACGCTCAGA… M     Y      GTTGGC…      10
# … wi

### Write to Insert Annotations File

In [11]:
insert_list %>% write_csv("../annotations/insert_annotations.csv")